# Legal Document Summarization: Fine-Tuning BART-Base on CivilSum
### Supervised Fine-Tuning and Held-Out Benchmark Evaluation on Indian Court Decisions

**Project:** Legal Document Summarization: A Zero-Shot Modular Agentic Workflow Approach  
**Starting Model:** `facebook/bart-base` (Hugging Face Transformers)  
**Dataset:** Official **CivilSum** Benchmark (SIGIR 2024)  
**Dataset Repository:** [https://github.com/ra-MANUJ-an/CivilSum](https://github.com/ra-MANUJ-an/CivilSum)  
**Dataset License:** Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (**CC BY-NC-SA 4.0**)  
**Target Hardware:** Google Colab Free T4 GPU or A100 GPU  

---

> **Research Integrity Notice:**  
> 1. This notebook trains an authentic neural checkpoint from `facebook/bart-base` on the verified CivilSum dataset.  
> 2. Do not fabricate dataset sizes, training loss logs, or ROUGE metrics. All metrics produced here are genuine outputs of your training run.  
> 3. Completed checkpoints are saved and downloaded to your local project directory at `models/checkpoints/civilsum_bart`.

## Step 0: Hardware Verification & Package Installation

In [ ]:
# Check GPU Availability
!nvidia-smi

import torch
assert torch.cuda.is_available(), "Please enable a GPU accelerator under Runtime > Change runtime type > T4 GPU."
print(f"CUDA Device Active: {torch.cuda.get_device_name(0)}")

In [ ]:
# Install dependencies for fine-tuning and evaluation
!pip install -q transformers datasets evaluate rouge-score accelerate gdown nltk pandas bert-score

## Step 1: Download Official CivilSum Dataset Splits
The dataset consists of 23,350 cases from the Supreme Court of India and High Courts:
- `train.csv`: 21,015 cases
- `validation.csv`: 1,168 cases
- `test.csv`: 1,167 cases

We download directly from the official Google Drive files specified in the author repository (`ra-MANUJ-an/CivilSum`).

In [ ]:
import os
import gdown

os.makedirs("data", exist_ok=True)

# Official Google Drive IDs from ra-MANUJ-an/CivilSum (SIGIR 2024)
splits = {
    "train.csv": "1WX5w-ClK82Yy916350bRp3hcuqab1D-x",
    "validation.csv": "1khcDFSca6zUv-dFQ4XaxMt8jRf-9PZbV",
    "test.csv": "1v0EM35CFWxFPc5D4bFxpf59mV_kEv0pw"
}

for filename, file_id in splits.items():
    out_path = os.path.join("data", filename)
    if not os.path.exists(out_path):
        print(f"Downloading official {filename}...")
        gdown.download(f"https://drive.google.com/uc?id={file_id}", out_path, quiet=False)
    else:
        print(f"{filename} already exists.")

## Step 2: Dataset Schema and Quality Inspection

In [ ]:
import pandas as pd

train_df = pd.read_csv("data/train.csv").dropna(subset=["text", "summary"])
val_df = pd.read_csv("data/validation.csv").dropna(subset=["text", "summary"])
test_df = pd.read_csv("data/test.csv").dropna(subset=["text", "summary"])

print("=== CivilSum Official Dataset Verification ===")
print(f"Train rows:      {len(train_df):,} (Expected ~21,015)")
print(f"Validation rows: {len(val_df):,} (Expected ~1,168)")
print(f"Test rows:       {len(test_df):,} (Expected ~1,167)")
print(f"Total cases:     {len(train_df) + len(val_df) + len(test_df):,}")
print(f"Columns:         {list(train_df.columns)}")

# Sample preview
print("\n--- Sample Judgment (First 300 characters) ---")
print(train_df.iloc[0]['text'][:300] + "...")
print("\n--- Sample Human Summary (First 200 characters) ---")
print(train_df.iloc[0]['summary'][:200] + "...")

## Step 3: Tail-Aware Tokenization Strategy
Indian court decisions documented in CivilSum feature a prominent **tail bias**, where the final judicial holding, ratio decidendi, and operative relief appear near the document end.  
We apply a domain-specific tokenizer pipeline preserving the initial 30% (parties and factual framing) and final 70% (arguments and disposition) within BART's 1024 token window.

In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = "facebook/bart-base"
MAX_SOURCE_LENGTH = 1024
MAX_TARGET_LENGTH = 350

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tail_aware_preprocess(batch):
    processed_texts = []
    for doc in batch["text"]:
        words = str(doc).split()
        if len(words) > 800:
            lead = " ".join(words[:240])
            tail = " ".join(words[-560:])
            processed_texts.append(f"{lead} {tail}")
        else:
            processed_texts.append(str(doc))

    model_inputs = tokenizer(
        processed_texts,
        max_length=MAX_SOURCE_LENGTH,
        truncation=True,
        padding="max_length",
    )

    labels = tokenizer(
        text_target=[str(s) for s in batch["summary"]],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding="max_length",
    )

    labels["input_ids"] = [
        [(token if token != tokenizer.pad_token_id else -100) for token in seq]
        for seq in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# For standard Colab training demonstration, you can fine-tune on a representative sample or full split
# Select subset for fast training or full train split depending on Colab timeout
TRAIN_SUBSET_SIZE = 4000  # Adjust as needed (or use len(train_df) for full training)

train_data = Dataset.from_pandas(train_df.head(TRAIN_SUBSET_SIZE)[["text", "summary"]])
val_data = Dataset.from_pandas(val_df.head(400)[["text", "summary"]])

print(f"Tokenizing {len(train_data)} train items and {len(val_data)} validation items...")
train_tokenized = train_data.map(tail_aware_preprocess, batched=True, remove_columns=["text", "summary"])
val_tokenized = val_data.map(tail_aware_preprocess, batched=True, remove_columns=["text", "summary"])

## Step 4: Model Initialization & Fine-Tuning Setup

In [ ]:
from transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer
import numpy as np
import evaluate

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 2) for k, v in result.items()}

OUTPUT_CHECKPOINT_DIR = "/content/civilsum_bart"

training_args = Seq2SeqTrainingArguments(
    output_dir="./checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    seed=42,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Step 5: Execute Training and Export Checkpoint

In [ ]:
# Execute Fine-Tuning
print("Starting Supervised Fine-Tuning on CivilSum...")
train_result = trainer.train()

# Save Model Checkpoint & Tokenizer
print(f"Exporting trained model weights to '{OUTPUT_CHECKPOINT_DIR}'...")
trainer.save_model(OUTPUT_CHECKPOINT_DIR)
tokenizer.save_pretrained(OUTPUT_CHECKPOINT_DIR)

# Write metadata file
import json, time
meta = {
    "base_model": MODEL_NAME,
    "dataset": "CivilSum (SIGIR 2024)",
    "train_samples": len(train_data),
    "val_samples": len(val_data),
    "training_loss": float(train_result.training_loss),
    "global_steps": int(train_result.global_step),
    "device": torch.cuda.get_device_name(0),
    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
}
with open(f"{OUTPUT_CHECKPOINT_DIR}/training_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Checkpoint successfully exported!")

## Step 6: Held-Out Benchmark Evaluation on Test Split
We evaluate genuine ROUGE-1, ROUGE-2, ROUGE-L, and BERTScore on the held-out `test.csv` split.

In [ ]:
from tqdm import tqdm
from rouge_score import rouge_scorer
import bert_score

EVAL_TEST_COUNT = 100  # Evaluate on first 100 held-out cases (or full 1,167)
eval_test = test_df.head(EVAL_TEST_COUNT)

model.eval()
model.to("cuda")

generated_summaries = []
reference_summaries = eval_test["summary"].tolist()

print(f"Generating predictions for {len(eval_test)} held-out cases...")
for doc in tqdm(eval_test["text"].tolist()):
    inputs = tokenizer(doc, max_length=MAX_SOURCE_LENGTH, truncation=True, return_tensors="pt").to("cuda")
    with torch.no_grad():
        summary_ids = model.generate(
            inputs["input_ids"],
            max_length=MAX_TARGET_LENGTH,
            min_length=80,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )
    pred = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    generated_summaries.append(pred.strip())

# Calculate ROUGE Scores
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
r1, r2, rl = [], [], []
for pred, ref in zip(generated_summaries, reference_summaries):
    scores = scorer.score(ref, pred)
    r1.append(scores["rouge1"].fmeasure * 100)
    r2.append(scores["rouge2"].fmeasure * 100)
    rl.append(scores["rougeL"].fmeasure * 100)

print("\n================ HELD-OUT TEST EVALUATION ================")
print(f"ROUGE-1 F1: {np.mean(r1):.2f}")
print(f"ROUGE-2 F1: {np.mean(r2):.2f}")
print(f"ROUGE-L F1: {np.mean(rl):.2f}")

# Calculate BERTScore
P, R, F1 = bert_score.score(generated_summaries, reference_summaries, lang="en", verbose=False)
print(f"BERTScore Precision: {P.mean().item()*100:.2f}")
print(f"BERTScore Recall:    {R.mean().item()*100:.2f}")
print(f"BERTScore F1:        {F1.mean().item()*100:.2f}")

## Step 7: Download Checkpoint for Local Streamlit Deployment
Download the zip file and extract it into your project folder:
`Legal-Document-Summarization/models/checkpoints/civilsum_bart`

In [ ]:
# Compress checkpoint folder for easy download
!zip -r civilsum_bart_checkpoint.zip /content/civilsum_bart

from google.colab import files
files.download("civilsum_bart_checkpoint.zip")
print("Downloaded checkpoint zip. Extract into models/checkpoints/civilsum_bart/")